In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
浙江水利 · 全省水位数据读取工具
读取 zhejiang_water_data.db 并导出为 CSV
"""

import sqlite3
import csv
import os
from pathlib import Path
from datetime import datetime

RUNTIME_ROOT = Path(r"E:\\AAAqian\\storm_surge_runtime_data")
DATA_PATH = RUNTIME_ROOT / "zhejiang_water_data"
DB_PATH = str(DATA_PATH / "zhejiang_water_data.db")
EXPORT_PATH = DATA_PATH / "exports"
EXPORT_PATH.mkdir(parents=True, exist_ok=True)


def check_database():
    """检查数据库结构和字段名"""
    if not os.path.exists(DB_PATH):
        print(f"❌ 数据库文件不存在: {DB_PATH}")
        print("   请先运行爬虫采集数据")
        return False
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 查看所有表
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = cursor.fetchall()
    print(f"📋 数据库中的表: {[t[0] for t in tables]}")
    
    # 查看表结构
    table_name = tables[0][0] if tables else None
    if table_name:
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        print(f"\n📋 表 '{table_name}' 的字段:")
        for col in columns:
            print(f"   {col[1]} ({col[2]})")
        
        # 判断字段名
        col_names = [col[1] for col in columns]
        county_field = "city_county" if "city_county" in col_names else "county" if "county" in col_names else None
        print(f"\n📌 市县字段名: {county_field}")
        
        # 统计总记录数
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        total = cursor.fetchone()[0]
        print(f"📊 总记录数: {total}")
        
        conn.close()
        return True
    
    conn.close()
    return False


def export_all_data(output_file="浙江省水位数据_全部.csv"):
    """导出全部数据"""
    if not os.path.exists(DB_PATH):
        print(f"❌ 数据库文件不存在: {DB_PATH}")
        return
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 获取所有列
    cursor.execute("PRAGMA table_info(zhejiang_water_data)")
    columns = [col[1] for col in cursor.fetchall()]
    
    # 查询所有数据
    cursor.execute(f"SELECT * FROM zhejiang_water_data ORDER BY CAST(water_level AS REAL) DESC")
    rows = cursor.fetchall()
    
    output_file = str(EXPORT_PATH / output_file)
    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)
    
    print(f"✅ 已导出 {len(rows)} 条 → {output_file}")
    conn.close()


def export_water_data(output_file="浙江省水位数据_纯净版.csv"):
    """导出有水位的记录"""
    if not os.path.exists(DB_PATH):
        print(f"❌ 数据库文件不存在: {DB_PATH}")
        return
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 先检查字段名
    cursor.execute("PRAGMA table_info(zhejiang_water_data)")
    col_names = [col[1] for col in cursor.fetchall()]
    
    # 确定市县字段名
    if "city_county" in col_names:
        county_field = "city_county"
    elif "county" in col_names:
        county_field = "county"
    else:
        county_field = None
        print("⚠ 未找到市县字段")
    
    county_col = f", {county_field}" if county_field else ""
    
    cursor.execute(f"""
        SELECT station_name{county_col}, water_level, time_str, crawl_time
        FROM zhejiang_water_data 
        WHERE water_level IS NOT NULL AND water_level != ''
        ORDER BY CAST(water_level AS REAL) DESC
    """)
    rows = cursor.fetchall()
    
    header = ["站名"]
    if county_field:
        header.append("市县")
    header.extend(["水位(m)", "上报时间", "采集时间"])
    
    output_file = str(EXPORT_PATH / output_file)
    with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)
    
    print(f"✅ 已导出 {len(rows)} 条 → {output_file}")
    conn.close()


def export_by_city(output_dir="按市县导出"):
    output_dir = str(EXPORT_PATH / output_dir)
    """按市县分别导出CSV"""
    if not os.path.exists(DB_PATH):
        print(f"❌ 数据库文件不存在: {DB_PATH}")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 检查字段名
    cursor.execute("PRAGMA table_info(zhejiang_water_data)")
    col_names = [col[1] for col in cursor.fetchall()]
    county_field = "city_county" if "city_county" in col_names else "county" if "county" in col_names else None
    
    if not county_field:
        print("⚠ 未找到市县字段，无法按市县导出")
        conn.close()
        return
    
    # 获取所有市县
    cursor.execute(f"SELECT DISTINCT {county_field} FROM zhejiang_water_data ORDER BY {county_field}")
    cities = [row[0] for row in cursor.fetchall()]
    
    total = 0
    for city in cities:
        cursor.execute(f"""
            SELECT station_name, water_level, time_str, crawl_time
            FROM zhejiang_water_data 
            WHERE {county_field} = ?
              AND water_level IS NOT NULL AND water_level != ''
            ORDER BY CAST(water_level AS REAL) DESC
        """, (city,))
        rows = cursor.fetchall()
        
        if rows:
            # 清洗文件名
            safe_name = city.replace("/", "_").replace("\\", "_").replace(" ", "")
            filename = os.path.join(output_dir, f"{safe_name}.csv")
            
            with open(filename, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.writer(f)
                writer.writerow(["站名", "水位(m)", "上报时间", "采集时间"])
                writer.writerows(rows)
            
            print(f"  {city}: {len(rows)} 条 → {filename}")
            total += len(rows)
    
    print(f"\n✅ 共导出 {total} 条到 {output_dir}/ 目录")
    conn.close()


def print_summary():
    #"""打印全省数据汇总"""
    if not os.path.exists(DB_PATH):
        print(f"❌ 数据库文件不存在: {DB_PATH}")
        return
    
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # 检查字段名
    cursor.execute("PRAGMA table_info(zhejiang_water_data)")
    col_names = [col[1] for col in cursor.fetchall()]
    county_field = "city_county" if "city_county" in col_names else "county" if "county" in col_names else "市县"
    
    # 总记录数
    cursor.execute("SELECT COUNT(*) FROM zhejiang_water_data")
    total = cursor.fetchone()[0]
    
    # 有水位的记录
    cursor.execute("SELECT COUNT(*) FROM zhejiang_water_data WHERE water_level IS NOT NULL AND water_level != ''")
    has_water = cursor.fetchone()[0]
    
    # 各市县站点数
    cursor.execute(f"""
        SELECT {county_field}, COUNT(*) as cnt
        FROM zhejiang_water_data
        GROUP BY {county_field}
        ORDER BY cnt DESC
    """)
    cities = cursor.fetchall()
    
    # 最高水位前5
    cursor.execute(f"""
        SELECT station_name, {county_field}, water_level, time_str
        FROM zhejiang_water_data
        WHERE water_level IS NOT NULL AND water_level != ''
        ORDER BY CAST(water_level AS REAL) DESC
        LIMIT 5
    """)
    top5 = cursor.fetchall()
    
    # 最低水位前5
    cursor.execute(f"""
        SELECT station_name, {county_field}, water_level, time_str
        FROM zhejiang_water_data
        WHERE water_level IS NOT NULL AND water_level != ''
        ORDER BY CAST(water_level AS REAL) ASC
        LIMIT 5
    """)
    bottom5 = cursor.fetchall()
    
    # 最新采集时间
    cursor.execute("SELECT MAX(crawl_time) FROM zhejiang_water_data")
    latest = cursor.fetchone()[0] or "无"
    
    conn.close()
    
    # 打印汇总
    print("\n" + "=" * 55)
    print("🌊 浙江省实时水情数据汇总")
    print("=" * 55)
    print(f"📊 总记录数:        {total}")
    print(f"📊 有水位数:        {has_water}")
    print(f"📊 涉及市县数:      {len(cities)}")
    print(f"🕐 最新采集时间:    {latest}")
    print()
    
    print("📍 各市县站点分布:")
    print(f"  {'市县':<12} {'站点数':>6}")
    print(f"  {'-'*20}")
    for city, cnt in cities:
        print(f"  {city:<12} {cnt:>6}")
    
    print(f"\n⬆ 最高水位 TOP 5:")
    print(f"  {'站名':<16} {'市县':<10} {'水位':>8}")
    print(f"  {'-'*38}")
    for row in top5:
        print(f"  {row[0]:<16} {row[1]:<10} {row[2]:>8}")
    
    print(f"\n⬇ 最低水位 TOP 5:")
    print(f"  {'站名':<16} {'市县':<10} {'水位':>8}")
    print(f"  {'-'*38}")
    for row in bottom5:
        print(f"  {row[0]:<16} {row[1]:<10} {row[2]:>8}")
    
    print("=" * 55)


def main():
    print("🌊 浙江省水位数据读取工具")
    print("=" * 40)
    
    # 检查数据库
    if not check_database():
        return
    
    while True:
        print("\n" + "=" * 40)
        print("请选择操作:")
        print("  1. 打印数据汇总")
        print("  2. 导出全部数据（CSV）")
        print("  3. 导出有水位的记录（纯净版）")
        print("  4. 按市县分别导出CSV")
        print("  5. 导出全部（一键三连）")
        print("  0. 退出")
        print("=" * 40)
        
        choice = input("请输入 [0-5]: ").strip()
        
        if choice == "0":
            print("👋 再见！")
            break
        elif choice == "1":
            print_summary()
        elif choice == "2":
            export_all_data()
        elif choice == "3":
            export_water_data()
        elif choice == "4":
            export_by_city()
        elif choice == "5":
            print("\n🚀 一键导出全部...")
            export_all_data()
            export_water_data()
            export_by_city()
            print("\n✅ 全部导出完成！")
        else:
            print("❌ 无效输入")


if __name__ == "__main__":
    main()


SyntaxError: unterminated triple-quoted string literal (detected at line 313) (1550675916.py, line 230)